<a href="https://colab.research.google.com/github/polmazon/tfm/blob/claude%2Ffestive-albattani-93veeb/scraper_competencia_honda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Web Scraping - ES


## 1. Install dependencies (run only in Colab)


In [ ]:
# Run this cell once in Google Colab (takes ~1 minute)
!pip install -q requests beautifulsoup4 "selenium>=4.15" pandas openai
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -q --fix-missing ./google-chrome-stable_current_amd64.deb
print("Dependencies installed successfully")

## 2. Imports and configuration


In [217]:
import requests
import time
import json
import re
import pandas as pd
from datetime import date
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from openai import OpenAI

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


In [ ]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

CAMPOS_OFERTA = [
    "marca",
    "modelo",
    "precio_vehiculo",
    "cuota_mensual",
    "plazo_meses",
    "entrada",
    "tin",
    "tae",
    "comision_apertura",
    "valor_residual",
    "importe_financiado",
    "tipo_financiacion",
    "banco_financiacion",
    "fecha_fin_oferta",
    "url",
    "fecha_extraccion"
]

print(f"API Key OpenAI cargada: {'OK' if OPENAI_API_KEY else 'ERROR — revisa los Secrets'}")

## 3. Scraping functions


In [ ]:
def scrape_estatico(url, reintentos=3, pausa=2):
    for intento in range(reintentos):
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
            response.raise_for_status()
            return response.text
        except requests.RequestException as e:
            print(f"  [intento {intento+1}/{reintentos}] Error en {url}: {e}")
            time.sleep(pausa * (intento + 1))
    return None


def crear_driver():
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(f"user-agent={HEADERS['User-Agent']}")
    # Selenium Manager descarga automáticamente el chromedriver compatible con el Chrome instalado
    return webdriver.Chrome(options=options)


def scroll_hasta_el_final(driver, pausas=8):
    for i in range(pausas):
        driver.execute_script("window.scrollBy(0, document.body.scrollHeight);")
        time.sleep(1.5)
    driver.execute_script("window.scrollTo(0, 0);")


def scroll_incremental(driver, paso=300, pausa=0.8):
    """Scroll step-by-step to trigger lazy-loaded content on each section."""
    # First pass: scroll to bottom repeatedly until height stabilises
    altura_prev = 0
    for _ in range(10):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        altura = driver.execute_script("return document.body.scrollHeight")
        if altura == altura_prev:
            break
        altura_prev = altura
    # Second pass: slow incremental scroll so each section enters viewport
    pos = 0
    while True:
        altura = driver.execute_script("return document.body.scrollHeight")
        if pos >= altura:
            break
        driver.execute_script(f"window.scrollTo(0, {pos});")
        time.sleep(pausa)
        pos += paso
    time.sleep(3)  # final wait for last elements to render


def _aceptar_cookies(driver):
    """Try to click common cookie accept buttons."""
    selectores = [
        "#onetrust-accept-btn-handler", "#accept-all-cookies",
        "button[id*=accept]", "button[class*=accept]",
        "button[id*=cookie]", "[data-testid*=accept]",
    ]
    for sel in selectores:
        try:
            btn = driver.find_element(By.CSS_SELECTOR, sel)
            if btn.is_displayed():
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(1)
                return True
        except Exception:
            pass
    return False


def scrape_dinamico(url, espera_extra=3, scroll=False, scroll_lento=False, wait_post_scroll=3):
    driver = crear_driver()
    try:
        driver.get(url)
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        time.sleep(espera_extra)
        _aceptar_cookies(driver)
        if scroll_lento:
            scroll_incremental(driver)
        elif scroll:
            scroll_hasta_el_final(driver)
        if wait_post_scroll > 0:
            time.sleep(wait_post_scroll)
        return driver.page_source
    except Exception as e:
        print(f"  Error Selenium en {url}: {e}")
        return None
    finally:
        driver.quit()


def html_a_texto(html, seccion_especial=None, max_chars_legal=4000,
                 pagina_listado=False, umbral_tin=6000, tomar_final=False,
                 modelo_hint=None):
    if not html:
        return ""
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "header", "noscript"]):
        tag.decompose()
    texto = soup.get_text(separator=" ", strip=True)
    texto = re.sub(r'\s+', ' ', texto)

    cabecera = texto[:2500]

    if seccion_especial:
        candidatos = seccion_especial if isinstance(seccion_especial, list) else [seccion_especial]
        for candidato in candidatos:
            posiciones = []
            inicio_busq = 0
            while True:
                p = texto.find(candidato, inicio_busq)
                if p == -1:
                    break
                posiciones.append(p)
                inicio_busq = p + 1
            if not posiciones:
                continue
            pos_elegida = posiciones[0]
            if modelo_hint and len(posiciones) > 1:
                hint_lower = modelo_hint.lower()
                hint_words = [w for w in hint_lower.split() if len(w) > 3]
                mejor = -1
                mejor_pos = posiciones[0]
                for p in posiciones:
                    contexto = texto[p:p + 300].lower()
                    coincidencias = sum(1 for w in hint_words if w in contexto)
                    if coincidencias > mejor:
                        mejor = coincidencias
                        mejor_pos = p
                pos_elegida = mejor_pos
            bloque = texto[pos_elegida:pos_elegida + max_chars_legal]
            n_bloques = len(posiciones)
            print(f"  Sección '{candidato[:40]}' encontrada ({n_bloques} bloque/s, usando pos {pos_elegida})")
            return cabecera + " [...] " + bloque
        print(f"  AVISO: Ninguna sección especial encontrada {[c[:30] for c in candidatos]}, usando fallback")

    if tomar_final:
        bloque = texto[-max_chars_legal:]
        print(f"  [FINAL] Extrayendo últimos {len(bloque)} chars de {len(texto)} totales")
        return cabecera + " [...] " + bloque

    pos = -1
    for kw in ["TIN:", "TIN :", "T.I.N", "TIN ", "TIN%", " TIN "]:
        p = texto.find(kw)
        if p != -1:
            pos = p
            break

    if pos != -1:
        if pagina_listado:
            # If seccion_especial is set, anchor from there instead of TIN
            if seccion_especial:
                candidatos = seccion_especial if isinstance(seccion_especial, list) else [seccion_especial]
                pos_ancla = -1
                for candidato in candidatos:
                    p = texto.find(candidato)
                    if p != -1 and (pos_ancla == -1 or p < pos_ancla):
                        pos_ancla = p
                if pos_ancla != -1:
                    bloque = texto[pos_ancla:]
                    print(f"  [LISTADO] Ancla encontrada en pos {pos_ancla} — extrayendo {len(bloque)} chars hasta el final")
                    return bloque[:max_chars_legal]
            bloque = texto[max(0, pos - 1500):]
            print(f"  [LISTADO] TIN en pos {pos} — extrayendo {len(bloque)} chars hasta el final")
            return cabecera + " [...] " + bloque
        elif pos < umbral_tin:
            inicio = max(0, pos - 1500)
            bloque = texto[inicio:inicio + max_chars_legal]
            print(f"  Texto enviado al LLM: {len(cabecera) + len(bloque)} chars (TIN en pos {pos})")
            return cabecera + " [...] " + bloque
        else:
            print(f"  TIN encontrado en pos {pos} (carrusel, ignorado) — usando cabecera")

    print(f"  Texto enviado al LLM: {len(cabecera)} chars (sin texto legal propio)")
    return cabecera


print("Funciones de scraping definidas")

## 4. LLM extraction


In [ ]:
client = OpenAI(api_key=OPENAI_API_KEY)

PROMPT_SISTEMA = """You are an expert in car financing offers in Spain.
Your task is to extract structured information from dealership web page texts.
ALWAYS return valid JSON with the specified fields.
If a field does not appear in the text, return null for that field.
Do not invent data. Only extract what is explicitly in the text."""

PROMPT_CAMPOS = """
For each offer return a JSON object with these fields:
- modelo: full commercial name of the model (including version and kW if present)
- tipo_combustible: "gasolina", "diésel", "híbrido", "híbrido enchufable", "eléctrico" or null
- precio_vehiculo: Vehicle RRP in € (number). Search in this order:
    1. "PVP al contado", "Precio al contado" or "Precio de adquisición al contado" — use that value
    2. If no separate cash price, look for "PVP recomendado financiando" or "PVP recomendado" — use that value
    NEVER use the financed amount or deposit as precio_vehiculo
- precio_financiar: Amount to finance in € (number). Rules:
    - If there are TWO different prices (one cash and one lower financed price), use the lower price
    - If there is only ONE price in the text, use that same price (same as precio_vehiculo)
    NEVER leave this field null if there is at least one price in the text
- cuota_mensual: Total monthly payment in € (number). If the payment is broken into components (e.g. finance payment + insurance), use the total sum
- plazo_meses: total contract duration in months (number)
- entrada: initial deposit in € (number, 0 if none)
- tin: TIN in % (number). Look for "TIN:" or "Tipo Deudor" followed by a percentage
- tae: TAE in % (number). Look for "TAE:" or "T.A.E." followed by a percentage
- comision_apertura: opening fee amount in € (number). Look for "Comisión de apertura" followed by an amount in €. Set to 0 ONLY if the text explicitly states it is free or "sin comisión"
- porcentaje_comision_apertura: comisión de apertura en % sobre el capital (número o null)
- valor_residual: última cuota o valor residual en € (número o null). Busca "Última cuota", "valor residual" o "cuota final"
- importe_financiado: capital total financiado en € (número). Busca "Capital financiado", "Importe financiado" o "Importe total del Crédito"
- tipo_financiacion: nombre exacto del producto financiero
- banco_financiacion: entidad bancaria que OFRECE el préstamo de financiación del vehículo.
    Busca frases como "Financiación ofrecida por", "sujeta a aprobación por parte de" DENTRO del bloque de condiciones financieras (donde aparecen TIN/TAE/cuotas).
    IGNORA completamente cualquier mención a entidades bancarias que aparezca en notas de mantenimiento, garantías u otros servicios posventa: esas menciones NO son el banco financiador del vehículo.
    Ejemplos válidos: "Santander Consumer Finance", "PSA Finance", "Volkswagen Financial Services", "Open Bank, S.A.". null si no aparece.
- fecha_fin_oferta: fecha límite en YYYY-MM-DD (string o null)

Devuelve SOLO este JSON:
{"ofertas": [ {...} ]}
"""


def _llamar_llm(trozo, marca, url, instruccion, max_tokens):
    prompt = f"""{instruccion}

Analiza el siguiente texto de la web de {marca} ({url}).
{PROMPT_CAMPOS}
TEXTO:
{trozo}"""
    respuesta = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        max_tokens=max_tokens,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": PROMPT_SISTEMA},
            {"role": "user", "content": prompt}
        ]
    )
    datos = json.loads(respuesta.choices[0].message.content)
    return [o for o in datos.get("ofertas", []) if o is not None]


def _anotar(ofertas, marca, url):
    for o in ofertas:
        o["marca"] = marca
        o["url"] = url
        o["fecha_extraccion"] = str(date.today())
    return ofertas


_MARCAS_PREFIJOS = {"volkswagen", "toyota", "peugeot", "renault", "nissan", "hyundai", "audi", "seat", "skoda", "honda", "mazda"}

def _fusionar_sin_duplicados(listas):
    """Fusiona listas deduplicando por modelo sin prefijo de marca."""
    vistos, resultado = set(), []
    for o in (o for lista in listas for o in lista):
        nombre = re.sub(r'\s+', ' ', (o.get("modelo") or "").strip().lower())
        palabras = nombre.split()
        if palabras and palabras[0] in _MARCAS_PREFIJOS:
            palabras = palabras[1:]
        key = " ".join(palabras)
        if key and key not in vistos:
            vistos.add(key)
            resultado.append(o)
    return resultado


def extraer_oferta_con_llm(texto, marca, url, filtro_producto=None, pagina_listado=False):
    slug = url.rstrip("/").split("/")[-1]
    pista_modelo = (slug.replace("-easy-plus", "").replace("-easy-renting", "")
                    .replace("-easy", "").replace("-", " ").title())

    try:
        if filtro_producto:
            productos = filtro_producto if isinstance(filtro_producto, list) else [filtro_producto]
            productos_str = " o ".join(f"'{p}'" for p in productos)
            instruccion = (
                f"IMPORTANT: The text may contain several legal blocks, one per product type. "
                f"Find the block for the product {productos_str} "
                f"(look for a heading mentioning {productos_str} or similar terms). "
                f"EXCLUDE any block for 'Leasing', 'Renting', 'Arrendamiento' or similar. "
                f"Extract ONLY the data from THAT block: PVP al contado (precio_vehiculo), "
                f"precio por financiar (precio_financiar), TIN, TAE, cuota mensual, entrada, "
                f"comisión de apertura, valor residual e importe financiado. "
                f"Return exactly ONE offer. Do not mix data from different blocks."
            )
            return _anotar(_llamar_llm(texto, marca, url, instruccion, 3000), marca, url)

        if pagina_listado:
            instruccion = (
                "Extrae TODAS las ofertas de financiación que encuentres en el texto legal. "
                "Cada bloque de condiciones legales corresponde a un modelo distinto. "
                "Si un mismo modelo tiene dos bloques (uno para Península/Baleares y otro para Canarias), extrae SOLO el de Península/Baleares. "
                "Devuelve una entrada por cada bloque/modelo con TIN o TAE propio."
            )
            n = len(texto)
            solape = 6000  # wide overlap to avoid splitting legal blocks
            tam_chunk = 20000  # ~3 models per chunk at 6.5k each
            num_chunks = max(3, -(-n // tam_chunk))  # ceiling division
            puntos = [i * (n // num_chunks) for i in range(num_chunks)]
            trozos = []
            for i, p in enumerate(puntos):
                inicio = max(0, p - solape)
                fin = puntos[i + 1] + solape if i + 1 < len(puntos) else n
                trozos.append(texto[inicio:fin])
            sizes_str = " + ".join(str(len(t)) for t in trozos)
            print(f"  [LISTADO] {num_chunks} llamadas: {sizes_str} chars")
            resultados = []
            for i, trozo in enumerate(trozos):
                ofertas_i = _llamar_llm(trozo, marca, url, instruccion, 5000)
                print(f"  Call {i+1}/{num_chunks}: {len(ofertas_i)} offers")
                resultados.append(_anotar(ofertas_i, marca, url))
                if i < len(trozos) - 1:
                    time.sleep(1)
            return _fusionar_sin_duplicados(resultados)

        instruccion = "Extrae la oferta de financiación principal que encuentres."
        return _anotar(_llamar_llm(texto, marca, url, instruccion, 3000), marca, url)

    except Exception as e:
        print(f"  Error LLM para {url}: {e}")
        return []


print("Cliente OpenAI (gpt-4o-mini, temperature=0) configurado")

## 5. Full pipeline: scraping + LLM extraction


In [ ]:
def procesar_url(url, marca, usar_selenium=False, scroll=False, scroll_lento=False,
                 wait_post_scroll=3, espera_extra=3,
                 seccion_especial=None, filtro_producto=None,
                 pagina_listado=False, max_chars_legal=4000, umbral_tin=6000,
                 tomar_final=False):
    print(f"Procesando: {marca} — {url}")
    html = scrape_dinamico(url, espera_extra=espera_extra, scroll=scroll, scroll_lento=scroll_lento, wait_post_scroll=wait_post_scroll) if usar_selenium else scrape_estatico(url)
    if not html:
        print(f"  Could not download {url}")
        return []
    slug = url.rstrip("/").split("/")[-1]
    pista_modelo_hint = (slug.replace("-easy-plus", "").replace("-easy-renting", "")
                         .replace("-easy", "").replace("-", " "))
    texto = html_a_texto(
        html,
        seccion_especial=seccion_especial,
        max_chars_legal=max_chars_legal,
        pagina_listado=pagina_listado,
        umbral_tin=umbral_tin,
        tomar_final=tomar_final,
        modelo_hint=pista_modelo_hint if filtro_producto else None
    )
    if not pagina_listado and not tomar_final and not any(kw in texto for kw in ["TIN:", "TIN :", "T.I.N", "TIN%", " TIN "]):
        print(f"  No TIN found in text — page has no financing offer, skipping")
        return []
    ofertas = extraer_oferta_con_llm(
        texto, marca, url,
        filtro_producto=filtro_producto,
        pagina_listado=pagina_listado
    )
    print(f"  Offers found: {len(ofertas)}")
    return ofertas



def descubrir_urls_ford():
    """Discover individual model offer URLs from ford.es/compra/promociones/particulares."""
    BASE = "https://www.ford.es"
    INDEX = BASE + "/compra/promociones/particulares"
    print(f"Discovering Ford URLs from {INDEX} ...")
    html = scrape_dinamico(INDEX, espera_extra=5, scroll=True)
    if not html:
        print("  Could not load Ford promotions page")
        return []
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(html, "html.parser")
    urls = []
    seen = set()
    EXCLUIR = ["/compra/promociones/particulares", "/compra/promociones"]
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if ("/compra/promociones/" in href
                and not any(href.rstrip("/") == ex for ex in EXCLUIR)
                and "renting" not in href.lower()):
            url_completa = href if href.startswith("http") else BASE + href
            if url_completa not in seen:
                seen.add(url_completa)
                urls.append(url_completa)
    print(f"  Ford URLs found: {len(urls)}")
    for u in urls:
        print(f"    {u}")
    return urls


def descubrir_urls_toyota():
    """Discover Toyota Easy offer URLs from toyota.es/promociones (excludes renting)."""
    BASE = "https://www.toyota.es"
    print(f"Discovering Toyota Easy URLs from {BASE}/promociones ...")
    html = scrape_estatico(f"{BASE}/promociones")
    if not html:
        print("  Could not download Toyota promotions page")
        return []
    soup = BeautifulSoup(html, "html.parser")
    urls = []
    EXCLUIR = ["/promociones/toyota-easy-plus", "/promociones/toyota-easy",
               "/promociones/toyota-easy-complet"]
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if ("/promociones/" in href
                and "easy" in href
                and "renting" not in href
                and not any(href.endswith(ex.split("/")[-1]) and "/finance-insurance/" not in href
                            for ex in EXCLUIR)
                and "/finance-insurance/" not in href):
            url_completa = href if href.startswith("http") else BASE + href
            if url_completa not in urls:
                urls.append(url_completa)
    print(f"  Easy URLs found: {len(urls)}")
    for u in urls:
        print(f"    {u}")
    return urls


def descubrir_urls_renault():
    """Discover car model URLs from promociones.renault.es/particulares/."""
    BASE = "https://promociones.renault.es"
    INDEX = BASE + "/particulares/"
    print(f"Discovering Renault URLs from {INDEX} ...")
    html = scrape_dinamico(INDEX, scroll=False)
    if not html:
        print("  No se pudo descargar la página de Renault")
        return []
    soup = BeautifulSoup(html, "html.parser")
    urls = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        partes = href.rstrip("/").split("/")
        if not ("/particulares/" in href
                and len([p for p in partes if p]) >= 2
                and not href.rstrip("/").endswith("/particulares")):
            continue
        # Excluir URLs con dígitos en el slug (son promociones genéricas, no modelos)
        slug = partes[-1] if partes[-1] else partes[-2]
        if any(c.isdigit() for c in slug):
            continue
        url_completa = href if href.startswith("http") else BASE + href
        if url_completa not in urls:
            urls.append(url_completa)
    print(f"  Renault URLs found: {len(urls)}")
    for u in urls:
        print(f"    {u}")
    return urls


def descubrir_urls_hyundai():
    """Extrae desde hyundai.com/es/es/modelos.html todas las URLs de modelos."""
    BASE = "https://www.hyundai.com"
    INDEX = BASE + "/es/es/modelos.html"
    print(f"Descubriendo URLs Hyundai desde {INDEX} ...")
    html = scrape_dinamico(INDEX, scroll=True)
    if not html:
        print("  No se pudo descargar la página de Hyundai")
        return []
    soup = BeautifulSoup(html, "html.parser")
    urls = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/es/es/modelos/" in href and href.endswith(".html"):
            segmento = href.rstrip("/").split("/es/es/modelos/")[-1]
            if ("/" in segmento or "configurador" in href
                    or "coches" in href or href.endswith("/modelos.html")):
                continue
            url_completa = href if href.startswith("http") else BASE + href
            if url_completa not in urls:
                urls.append(url_completa)
    print(f"  URLs Hyundai encontradas: {len(urls)}")
    for u in urls:
        print(f"    {u}")
    return urls


def descubrir_urls_mazda():
    """Extrae desde mazda.es/promociones/promociones-actuales/ las URLs de modelos con oferta."""
    BASE = "https://www.mazda.es"
    INDEX = BASE + "/promociones/promociones-actuales/"
    print(f"Descubriendo URLs Mazda desde {INDEX} ...")
    html = scrape_dinamico(INDEX, scroll=True)
    if not html:
        print("  No se pudo descargar la página de Mazda")
        return []
    soup = BeautifulSoup(html, "html.parser")
    urls = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        # Solo aceptar URLs bajo /promociones-actuales/ (excluye /como-comprar/ y otras secciones)
        if "/promociones-actuales/" not in href:
            continue
        url_completa = href if href.startswith("http") else BASE + href
        if not url_completa.startswith(BASE):
            continue
        slug = url_completa.rstrip("/").split("/")[-1]
        # Excluir la página índice
        if slug in ("promociones-actuales", ""):
            continue
        if url_completa not in urls:
            urls.append(url_completa)
    print(f"  URLs Mazda encontradas: {len(urls)}")
    for u in urls:
        print(f"    {u}")
    return urls



def descubrir_urls_nissan():
    """Extrae desde nissan.es/ofertas.html las URLs de detalle de cada oferta."""
    BASE_URL = "https://www.nissan.es/ofertas.html"
    LISTING_URL = BASE_URL + "#category=PARTICULARES"
    print(f"Descubriendo URLs Nissan desde {LISTING_URL} ...")
    driver = crear_driver()
    urls = []
    try:
        driver.get(LISTING_URL)
        time.sleep(5)

        def contar_ofertas():
            """Cuenta las tarjetas de oferta actualmente en el DOM."""
            js = "return document.querySelectorAll('[href*=offerId]').length;"
            return driver.execute_script(js) or 0

        def buscar_boton_ver_mas():
            """Busca el botón 'Ver más' de carga por texto exacto (no 'Ver detalles')."""
            candidatos = driver.find_elements(By.TAG_NAME, "button")
            candidatos += driver.find_elements(By.XPATH, "//a[not(contains(@href,'offerId'))]")
            for el in candidatos:
                try:
                    texto = el.text.strip().upper()
                    if texto in ("VER MÁS", "VER MAS", "CARGAR MÁS", "CARGAR MAS", "VER MÁS OFERTAS"):
                        if el.is_displayed():
                            return el
                except Exception:
                    pass
            return None

        # Fase 1: clicar "Ver más" por texto exacto
        clicks = 0
        n_antes = contar_ofertas()
        for _ in range(30):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
            boton = buscar_boton_ver_mas()
            if not boton:
                break
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", boton)
            time.sleep(1)
            driver.execute_script("arguments[0].click();", boton)
            clicks += 1
            time.sleep(3)
            n_despues = contar_ofertas()
            print(f"  Click 'Ver mas' #{clicks} — offers in DOM: {n_antes} → {n_despues}")
            n_antes = n_despues

        # Fase 2: scroll incremental como fallback (para lazy-load sin botón)
        altura_total = driver.execute_script("return document.body.scrollHeight")
        paso = 400
        pos = 0
        while pos < altura_total:
            driver.execute_script(f"window.scrollTo(0, {pos});")
            time.sleep(0.4)
            pos += paso
            altura_total = driver.execute_script("return document.body.scrollHeight")

        time.sleep(2)
        n_final = contar_ofertas()
        print(f"  Total ofertas en DOM tras scroll: {n_final} (clicks 'Ver mas': {clicks})")

        # Extracción de offerIds via JS
        js_extract = (
            "var ids = [];"
            "document.querySelectorAll('[href*=offerId]').forEach(function(el){"
            "  var m = (el.getAttribute('href')||'').match(/offerId=([A-Za-z0-9_-]+)/);"
            "  if(m) ids.push(m[1]);"
            "});"
            "return ids;"
        )
        offer_ids_js = driver.execute_script(js_extract) or []

        # Fallback regex sobre el source
        source = driver.page_source
        offer_ids_regex = re.findall(r'offerId[=:"\s]+([A-Za-z0-9_\-]+)', source)

        seen = set()
        for oid in offer_ids_js + offer_ids_regex:
            if oid and oid not in seen and oid != "currentOfferId":
                seen.add(oid)
                urls.append(f"{BASE_URL}#category=PARTICULARES&offerId={oid}")

        print(f"  offerIds Nissan encontrados: {len(urls)}")
        for u in urls:
            print(f"    {u}")
    finally:
        driver.quit()
    return urls




def descubrir_urls_volvo():
    """Discover individual model promotion URLs from volvocars.com/es/promotions/."""
    BASE = "https://www.volvocars.com"
    INDEX = BASE + "/es/promotions/"
    print(f"Discovering Volvo URLs from {INDEX} ...")
    driver = crear_driver()
    urls = []
    seen = set()
    try:
        driver.get(INDEX)
        time.sleep(6)
        _aceptar_cookies(driver)
        scroll_incremental(driver, paso=400, pausa=0.8)
        time.sleep(3)

        # Extract all hrefs via JS
        all_hrefs = driver.execute_script(
            "return Array.from(document.querySelectorAll('a[href]')).map(a => a.getAttribute('href'));"
        ) or []

        # Debug: show sample hrefs
        volvo_hrefs = [h for h in all_hrefs if h and "volvo" in h.lower() or (h and "promotions" in h.lower())]
        print(f"  All hrefs count: {len(all_hrefs)}, volvo/promo hrefs: {len(volvo_hrefs)}")
        for h in volvo_hrefs[:20]:
            print(f"    href: {h}")

        # Accept /es/promotions/... and also /es/cars/.../offers or similar
        EXCLUDE = ["/es/promotions", "/es/promotions/"]
        for href in all_hrefs:
            if not href:
                continue
            # Match /es/promotions/<something>
            if ("/es/promotions/" in href
                    and href.rstrip("/") not in ["/es/promotions"]
                    and href.rstrip("/") != BASE + "/es/promotions"):
                url_completa = href if href.startswith("http") else BASE + href
                if url_completa not in seen:
                    seen.add(url_completa)
                    urls.append(url_completa)

        # Fallback: try page source regex for promotion slugs
        if not urls:
            import re
            source = driver.page_source
            slugs = re.findall(r"""["'/]((?:es/)?promotions/[a-z0-9][a-z0-9\-/]+)["']""", source)
            for slug in slugs:
                if slug.rstrip("/") in ("es/promotions", "promotions"):
                    continue
                url_completa = BASE + "/" + slug.lstrip("/")
                if url_completa not in seen:
                    seen.add(url_completa)
                    urls.append(url_completa)
            if urls:
                print(f"  Found {len(urls)} URLs via regex fallback")

    finally:
        driver.quit()

    print(f"  Volvo URLs found: {len(urls)}")
    for u in urls:
        print(f"    {u}")
    return urls


## 6. Competitor URLs


In [ ]:
COMPETENCIA = {
    "TOYOTA": {
        "selenium": True, "scroll": True,
        "seccion_especial": ["Precio correspondiente a", "Precio por financiar:", "Toyota Easy Plus:", "Toyota Easy Plus", "Oferta financiera con el producto Toyota Easy"], "filtro_producto": ["Easy Plus", "Easy"],
        "pagina_listado": False, "max_chars_legal": 10000, "tomar_final": True,
        "umbral_tin": 999999,
        "urls": []  # se auto-descubren desde toyota.es/promociones
    },
    "FORD": {
        "selenium": True, "scroll": True, "scroll_lento": False,
        "espera_extra": 5, "wait_post_scroll": 3,
        "seccion_especial": ["Información legal importante", "Pen. y Bal", "Península y Baleares"],
        "filtro_producto": None,
        "pagina_listado": False, "max_chars_legal": 20000,
        "tomar_final": True, "umbral_tin": 999999,
        "urls": []  # auto-discovered from ford.es/compra/promociones/particulares
    },
    "VOLKSWAGEN": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": True, "max_chars_legal": 60000,
        "urls": ["https://www.volkswagen.es/es/ofertas.html"]
    },
    "PEUGEOT": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": True, "max_chars_legal": 60000,
        "urls": ["https://www.peugeot.es/comprar/ofertas-del-momento.html"]
    },
    "RENAULT": {
        "selenium": True, "scroll": False,
        "seccion_especial": "CONDICIONES LEGALES PARA PENÍNSULA Y BALEARES",
        "filtro_producto": None,
        "pagina_listado": False, "max_chars_legal": 4000,
        "urls": []  # se auto-descubren desde promociones.renault.es/particulares/
    },
    "NISSAN": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": "Flex 4D",
        "pagina_listado": False, "max_chars_legal": 10000,
        "tomar_final": True,
        "umbral_tin": 999999,
        "urls": []  # auto-discovered from nissan.es/ofertas.html
    },
    "HYUNDAI": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": False, "max_chars_legal": 8000,
        "umbral_tin": 20000, "tomar_final": True,
        "urls": []  # se auto-descubren desde hyundai.com/es/es/modelos.html
    },
    "AUDI": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": True, "max_chars_legal": 60000,
        "urls": ["https://www.audi.es/es/compra/promociones/"]
    },
    "HONDA": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": True, "max_chars_legal": 60000,
        "urls": ["https://www.honda.es/cars/offers.html"]
    },
    "MAZDA": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": False, "max_chars_legal": 8000,
        "umbral_tin": 999999, "tomar_final": True,
        "urls": []  # se auto-descubren desde mazda.es/promociones/promociones-actuales/
    },
    "VOLVO": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": False, "max_chars_legal": 10000,
        "tomar_final": True, "umbral_tin": 999999,
        "urls": []  # auto-discovered from volvocars.com/es/promotions/
    }
}

print(f"Configuradas {len(COMPETENCIA)} marcas")

## 7. Run scraping


In [ ]:
MARCAS_A_EJECUTAR = ["TOYOTA", "FORD", "VOLKSWAGEN", "PEUGEOT", "RENAULT", "NISSAN", "HYUNDAI", "AUDI", "MAZDA", "HONDA", "VOLVO"]
# Para probar una sola marca: MARCAS_A_EJECUTAR = ["MAZDA"]

todas_las_ofertas = []

for marca in MARCAS_A_EJECUTAR:
    config = COMPETENCIA[marca]
    print(f"\n{'='*50}\nMARCA: {marca}\n{'='*50}")

    # Auto-descubrimiento de URLs por marca
    if marca == "TOYOTA":
        urls = descubrir_urls_toyota()
        if not urls:
            print("  Fallback a URLs hardcodeadas")
            urls = config["urls"]
    elif marca == "RENAULT":
        urls = descubrir_urls_renault()
        if not urls:
            print("  Fallback a URLs hardcodeadas")
            urls = config["urls"]
    elif marca == "FORD":
        urls = descubrir_urls_ford()
    elif marca == "NISSAN":
        urls = descubrir_urls_nissan()
        if not urls:
            print("  Fallback a URLs hardcodeadas")
            urls = config["urls"]
    elif marca == "HYUNDAI":
        urls = descubrir_urls_hyundai()
        if not urls:
            print("  Fallback a URLs hardcodeadas")
            urls = config["urls"]
    elif marca == "MAZDA":
        urls = descubrir_urls_mazda()
        if not urls:
            print("  Fallback a URLs hardcodeadas")
            urls = config["urls"]
    elif marca == "VOLVO":
        urls = descubrir_urls_volvo()
        if not urls:
            print("  Fallback a URLs hardcodeadas")
            urls = config["urls"]
    else:
        urls = config["urls"]

    if not urls:
        print(f"  Sin URLs para {marca}, saltando.")
        continue

    for url in urls:
        ofertas = procesar_url(
            url, marca,
            usar_selenium=config["selenium"],
            scroll=config.get("scroll", False),
            scroll_lento=config.get("scroll_lento", False),
            wait_post_scroll=config.get("wait_post_scroll", 3),
            espera_extra=config.get("espera_extra", 3),
            seccion_especial=config.get("seccion_especial"),
            filtro_producto=config.get("filtro_producto"),
            pagina_listado=config.get("pagina_listado", False),
            max_chars_legal=config.get("max_chars_legal", 4000),
            umbral_tin=config.get("umbral_tin", 6000),
            tomar_final=config.get("tomar_final", False)
        )
        todas_las_ofertas.extend(ofertas)
        time.sleep(2)

print(f"\n{'='*50}")
print(f"RESUMEN: {len(todas_las_ofertas)} ofertas brutas extraídas")
marcas_con_datos = set(o["marca"] for o in todas_las_ofertas)
print(f"Marcas CON datos: {marcas_con_datos}")
marcas_sin_datos = set(MARCAS_A_EJECUTAR) - marcas_con_datos
if marcas_sin_datos:
    print(f"Marcas SIN datos: {marcas_sin_datos}")

In [ ]:
df_bruto = pd.DataFrame(todas_las_ofertas)

if df_bruto.empty:
    print("No offers extracted.")
else:
    df = (
        df_bruto
        .drop_duplicates(subset=["url", "modelo"], keep="first")
        .reset_index(drop=True)
    )

    # Drop rows without TIN (incomplete financial data)
    mask_con_datos = df["tin"].notna()
    descartadas = (~mask_con_datos).sum()
    if descartadas > 0:
        print(f"  Dropping {descartadas} rows without TIN")
    df = df[mask_con_datos].reset_index(drop=True)

    # Classify financing type: PCP if residual value > 0, HP otherwise
    df["tipo_financiacion"] = df.apply(
        lambda r: "PCP" if (pd.notna(r.get("valor_residual")) and float(r.get("valor_residual") or 0) > 0)
                  else "HP",
        axis=1
    )

    # Drop rows where prices look like they were extracted in k€ instead of €
    if "precio_vehiculo" in df.columns:
        pv_check = pd.to_numeric(df["precio_vehiculo"], errors="coerce")
        bad_price = pv_check.notna() & (pv_check < 1000)
        if bad_price.any():
            print(f"  Dropping {bad_price.sum()} rows with implausible vehicle price (<1000€)")
            df = df[~bad_price].reset_index(drop=True)

    # Fix prices: if precio_financiar > precio_vehiculo the LLM picked a "from" price
    # from the carousel as RRP. In that case set precio_vehiculo = precio_financiar and promocion = 0.
    if "precio_vehiculo" in df.columns and "precio_financiar" in df.columns:
        pv = pd.to_numeric(df["precio_vehiculo"], errors="coerce")
        pf = pd.to_numeric(df["precio_financiar"], errors="coerce")
        mask_invertido = pf > pv
        if mask_invertido.any():
            print(f"  Fixing {mask_invertido.sum()} rows where precio_financiar > precio_vehiculo")
            df.loc[mask_invertido, "precio_vehiculo"] = pf[mask_invertido]
            pv = pd.to_numeric(df["precio_vehiculo"], errors="coerce")
            pf = pd.to_numeric(df["precio_financiar"], errors="coerce")
        df["promocion_financiacion"] = (pv - pf).round(2).fillna(0)

    # Normalise term: 48→49, 36→37 (commercial period convention)
    if "plazo_meses" in df.columns:
        df["plazo_meses"] = pd.to_numeric(df["plazo_meses"], errors="coerce")
        df["plazo_meses"] = df["plazo_meses"].replace({48: 49, 36: 37})

    CAMPOS_ORDENADOS = [
        "marca", "modelo", "tipo_combustible", "precio_vehiculo", "precio_financiar",
        "promocion_financiacion", "cuota_mensual", "plazo_meses", "entrada", "tin", "tae",
        "porcentaje_comision_apertura", "comision_apertura",
        "valor_residual", "importe_financiado", "tipo_financiacion",
        "fecha_fin_oferta", "banco_financiacion", "url", "fecha_extraccion"
    ]
    cols = [c for c in CAMPOS_ORDENADOS if c in df.columns]
    df = df[cols]

    pd.set_option("display.max_columns", None)
    pd.set_option("display.max_rows", 100)

    print(f"Raw offers: {len(df_bruto)} → after dedup and filter: {len(df)}")
    print(f"\nModels per brand:")
    print(df.groupby("marca")["modelo"].count().to_string())
    display(df)

In [224]:
# Export to CSV
nombre_archivo = f"ofertas_competencia_{date.today().strftime('%Y%m%d')}.csv"
df.to_csv(nombre_archivo, index=False, encoding="utf-8-sig")
print(f"Saved to: {nombre_archivo}")

# To download in Colab:
# from google.colab import files
# files.download(nombre_archivo)

Guardado en: ofertas_competencia_20260614.csv


In [225]:
# Comparative summary by brand
if not df.empty and "marca" in df.columns:
    resumen = df.groupby("marca").agg(
        num_ofertas=("modelo", "count"),
        tin_medio=("tin", "mean"),
        tae_medio=("tae", "mean"),
        cuota_min=("cuota_mensual", "min"),
        cuota_max=("cuota_mensual", "max"),
        mean_com_apertura=("porcentaje_comision_apertura", "mean")
    ).round(2)
    print(resumen)

            num_ofertas  tin_medio  tae_medio  cuota_min  cuota_max
marca                                                              
TOYOTA               18       7.03       8.24       99.0      495.0
VOLKSWAGEN           21       6.95       8.71      100.0      350.0


## 8. Export results to the price analysis Excel

This cell reads the CSV generated by the scraper and updates the Excel template with competitor data, tab by tab. **It does not modify any previous cells.** Upload the file `02._CAR_JUNE_2026__PRICE_COMPETENCE_ANALISIS.xlsx` to Colab before running.


In [ ]:
# ============================================================
# CELL 8 — Export data to the price analysis Excel
# - Blue:   Honda columns with new data different from the Excel
# - Yellow: competitor columns with new data different from the Excel
# - Red:    cell with no data found in the CSV
# - No colour: data found but identical to what was already in the Excel
# ============================================================
!pip install -q openpyxl rapidfuzz

import os, glob
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
import pandas as pd
from rapidfuzz import process, fuzz

FILL_HONDA    = PatternFill(start_color="ADD8E6", end_color="ADD8E6", fill_type="solid")
FILL_UPDATED  = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
FILL_NO_DATA  = PatternFill(start_color="FF4444", end_color="FF4444", fill_type="solid")
FILL_NONE     = PatternFill(fill_type=None)  # no fill

# --- 1. CSV ---
csvs = sorted(glob.glob("ofertas_competencia_*.csv"), reverse=True)
if csvs:
    csv_path = csvs[0]
    print(f"CSV found automatically: {csv_path}")
else:
    print("Select the CSV from your computer:")
    from google.colab import files
    csv_path = list(files.upload().keys())[0]

df_csv = pd.read_csv(csv_path)
print(f"CSV loaded: {len(df_csv)} rows")

# --- 2. Excel template ---
xl_found = next(
    (f for f in sorted(glob.glob("*.xlsx"))
     if any(k in f.lower() for k in ["price", "car", "analisis"])),
    None
)
if xl_found:
    EXCEL_TEMPLATE = xl_found
    print(f"Excel found: {EXCEL_TEMPLATE}")
else:
    print("Select the Excel template from your computer:")
    from google.colab import files
    EXCEL_TEMPLATE = list(files.upload().keys())[0]

wb = load_workbook(EXCEL_TEMPLATE)
print(f"Available sheets: {wb.sheetnames}")

# --- 3. Motorisation type detection ---
TIPOS_MOTOR = [
    ("electrico", ["bev","eléctric","electric","e-2008","e-208","e-308","e-3008",
                   "e-408","e-5008","ioniq","zoe","megane e","ariya","leaf",
                   "micra","inster","mazda6e","kona eléctrico"]),
    ("phev",      ["phev","plug-in","plug in","recargable","e:phev",
                   "rav4 plug","tucson phev","santa fe phev"]),
    ("hibrido",   ["hybrid","híbrido","hibrido","hev","self-charging","i-mmd",
                   "mild hybrid","mhev","full hybrid","e-tech","e-power",
                   "e-skyactiv","48v","yaris","corolla","rav4","jazz","hr-v",
                   "crv","cr-v","crosstar","zr-v","civic","prelude",
                   "captur","symbioz","austral","juke","qashqai",
                   "kona híbrido","tucson híbrido","santa fe híbrido"]),
    ("gasolina",  ["tsi","gdi","1.0t","1.5t","tfsi","gasolina","petrol","turbo",
                   "tce","puretech","dig-t","sce","dci","tdi","diesel","diésel",
                   "mpi","eco-g","glp"]),
]

def detectar_tipo(texto):
    if not texto: return None
    t = str(texto).lower()
    for tipo, kws in TIPOS_MOTOR:
        if any(k in t for k in kws): return tipo
    return None

df_csv["_tipo"] = df_csv["modelo"].apply(detectar_tipo)
df_honda = df_csv[df_csv["marca"].str.upper() == "HONDA"].copy()

# --- 4. Búsqueda fuzzy con filtro de motorización ---
def buscar(df, nombre, plazo_ref=None):
    tipo = detectar_tipo(nombre)
    sub = df[df["_tipo"] == tipo] if tipo else df
    if sub.empty: sub = df
    res = process.extractOne(
        nombre, sub["modelo"].fillna("").tolist(),
        scorer=fuzz.token_set_ratio, score_cutoff=75
    )
    aviso = ""
    if res is None and tipo:
        res = process.extractOne(
            nombre, df["modelo"].fillna("").tolist(),
            scorer=fuzz.token_set_ratio, score_cutoff=75
        )
        if res is None: return None, 0, None, tipo, ""
        aviso = " ⚠️ (distinto tipo motor)"
        sub = df
    elif res is None:
        return None, 0, None, tipo, ""
    sub2 = sub[sub["modelo"].fillna("") == res[0]].copy()
    if plazo_ref and "plazo_meses" in sub2.columns:
        pp = sub2[sub2["plazo_meses"] == plazo_ref]
        if not pp.empty: sub2 = pp
    if "cuota_mensual" in sub2.columns:
        sub2 = sub2.sort_values("cuota_mensual")
    return sub2.iloc[0], res[1], res[0], tipo, aviso

def norm_pct(v):
    try: f = float(v); return f / 100 if f > 1 else f
    except: return v

def valores_iguales(v_nuevo, v_excel):
    """Compara dos valores ignorando diferencias de tipo y redondeo mínimo."""
    if v_nuevo is None and v_excel is None: return True
    if v_nuevo is None or v_excel is None: return False
    try:
        return round(float(v_nuevo), 4) == round(float(v_excel), 4)
    except:
        return str(v_nuevo).strip().lower() == str(v_excel).strip().lower()

# --- 5. Campos: fila → (columna CSV, es_porcentaje) ---
CAMPOS = {
    5:  ("cuota_mensual",               False),
    6:  ("entrada",                     False),
    7:  ("tin",                         True),
    8:  ("porcentaje_comision_apertura", True),
    9:  ("tae",                         True),
    10: ("plazo_meses",                 False),
    11: ("valor_residual",              False),
    13: ("precio_financiar",            False),
}

def escribir_campos(ws, col_idx, oferta, fill_cambio, model_name=None):
    """
    - Dato encontrado y distinto al Excel → escribe + fill_cambio (azul/amarillo)
    - Dato encontrado e igual al Excel    → escribe + sin relleno
    - Sin dato para ese campo             → celda roja (valor intacto)
    - Fila 19: nombre del modelo del CSV
    """
    for fila, (campo, es_pct) in CAMPOS.items():
        c = ws.cell(row=fila, column=col_idx)
        if campo in oferta.index and pd.notna(oferta[campo]):
            valor = norm_pct(oferta[campo]) if es_pct else oferta[campo]
            if valores_iguales(valor, c.value):
                c.value = valor
                c.fill = FILL_NONE   # igual que antes → sin resaltar
            else:
                c.value = valor
                c.fill = fill_cambio  # cambió → amarillo o azul
        else:
            c.fill = FILL_NO_DATA    # sin dato → rojo (no toca el valor)
    if model_name is not None:
        ws.cell(row=19, column=col_idx).value = model_name

# --- 6. Columnas Honda por pestaña (col openpyxl: D=4, I=9) ---
HONDA_COLS = {
    "MONTHLY JAZZ":     [{"col": 4, "modelo": "Jazz"},
                         {"col": 9, "modelo": "Jazz Crosstar"}],
    "MONTHLY HRV":      [{"col": 4, "modelo": "HR-V"}],
    "MONTHLY CIVIC":    [{"col": 4, "modelo": "Civic"}],
    "MONTHLY ZRV":      [{"col": 4, "modelo": "ZR-V"}],
    "MONTHLY CRV FHEV": [{"col": 4, "modelo": "CR-V"}],
    "MONTHLY CRV PHEV": [{"col": 4, "modelo": "CR-V e:PHEV"}],
}

TABS_PLAZO = {
    "MONTHLY JAZZ":     36,
    "MONTHLY HRV":      36,
    "MONTHLY CIVIC":    36,
    "MONTHLY ZRV":      36,
    "MONTHLY CRV FHEV": 36,
    "MONTHLY CRV PHEV": 36,
}
sheet_map = {s.upper(): s for s in wb.sheetnames}

# --- 7. Rellenar cada pestaña ---
for tab_key, plazo_ref in TABS_PLAZO.items():
    real = sheet_map.get(tab_key)
    if not real:
        print(f"⚠️  '{tab_key}' no encontrada")
        continue
    ws = wb[real]
    print(f"\n📋 {real}")

    # 7a. Columnas Honda (azul si cambia)
    if not df_honda.empty:
        for h in HONDA_COLS.get(tab_key, []):
            oferta, score, match, tipo, aviso = buscar(df_honda, h["modelo"], plazo_ref)
            if oferta is not None:
                escribir_campos(ws, h["col"], oferta, FILL_HONDA, model_name=match)
                print(f"  🔵 Honda '{h['modelo']}' → '{match}' ({score}%)")
            else:
                for fila in CAMPOS:
                    ws.cell(row=fila, column=h["col"]).fill = FILL_NO_DATA
                print(f"  ❌ Honda '{h['modelo']}' — sin datos en CSV")
    else:
        print("  ℹ️  Sin filas Honda en el CSV (ejecuta scraper con HONDA incluido)")

    # 7b. Columnas competencia (amarillo si cambia)
    vacias = 0; actualizados = 0
    for col_idx in range(5, 31):
        cn = ws.cell(row=4, column=col_idx)
        nombre = cn.value
        if not nombre:
            vacias += 1
            if vacias >= 3: break
            continue
        vacias = 0

        oferta, score, match, tipo, aviso = buscar(df_csv, str(nombre), plazo_ref)
        tipo_tag = f"[{tipo}]" if tipo else "[?]"

        if oferta is not None:
            escribir_campos(ws, col_idx, oferta, FILL_UPDATED, model_name=match)
            actualizados += 1
            print(f"  ✅ {tipo_tag} '{nombre}' → '{match}' ({score}%){aviso}")
        else:
            for fila in CAMPOS:
                ws.cell(row=fila, column=col_idx).fill = FILL_NO_DATA
            print(f"  ❌ {tipo_tag} '{nombre}' — sin coincidencia en CSV")

    print(f"  → {actualizados} competidores actualizados")

# --- 8. Guardar y descargar ---
OUTPUT = "02._CAR_PRICE_COMPETENCE_ACTUALIZADO.xlsx"
wb.save(OUTPUT)
print(f"\n✅ Excel guardado: {OUTPUT}")
try:
    from google.colab import files
    files.download(OUTPUT)
    print("📥 Descarga iniciada automáticamente.")
except Exception:
    print(f"Descarga manual: panel de archivos de Colab → '{OUTPUT}'.")
